<a href="https://colab.research.google.com/github/amitadhainje/GenerativeAI/blob/main/Training_YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install kaggle
!pip install kaggle -q

# Upload your kaggle.json
from google.colab import files
files.upload()  # upload kaggle.json here

# Move it to the right location
import os
os.makedirs("/root/.config/kaggle", exist_ok=True)
!cp kaggle.json /root/.config/kaggle/kaggle.json
!chmod 600 /root/.config/kaggle/kaggle.json
print("Kaggle API ready!")

Saving kaggle.json to kaggle.json
Kaggle API ready!


In [2]:
KAGGLE_DATASET = "marquis03/plants-classification"  # ← paste your slug here

!kaggle datasets download -d {KAGGLE_DATASET} --unzip -p /content/kaggle_data
print("Download complete!")

Dataset URL: https://www.kaggle.com/datasets/marquis03/plants-classification
License(s): apache-2.0
100% 1.34G/1.34G [00:15<00:00, 95.8MB/s]

Download complete!


In [3]:
import os

for root, dirs, files in os.walk("/content/kaggle_data"):
    level = root.replace("/content/kaggle_data", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:  # only show files up to 3 levels deep
        for f in files[:5]:  # show first 5 files per folder
            print(f"{indent}  {f}")

kaggle_data/
  train.csv
  test.csv
  val.csv
  val/
    classname.txt
    peperchili/
      peperchili721.jpg
      peperchili729.jpg
      peperchili773.jpg
      peperchili743.jpg
      peperchili750.jpg
    sweetpotatoes/
      sweetpotatoes715.jpg
      sweetpotatoes759.jpg
      sweetpotatoes793.jpg
      sweetpotatoes746.jpg
      sweetpotatoes756.jpg
    guava/
      guava713.jpg
      guava761.jpg
      guava741.jpg
      guava752.jpg
      guava757.jpg
    orange/
      orange759.jpg
      orange724.jpg
      orange711.jpg
      orange709.jpg
      orange758.jpg
    corn/
      corn758.jpg
      corn799.jpg
      corn754.jpg
      corn710.jpg
      corn747.jpg
    aloevera/
      aloevera723.jpg
      aloevera797.jpg
      aloevera783.jpg
      aloevera792.jpg
      aloevera754.jpg
    paddy/
      paddy702.jpg
      paddy744.jpg
      paddy712.jpg
      paddy725.jpg
      paddy780.jpg
    cucumber/
      cucumber788.jpg
      cucumber705.jpg
      cucumber727.jpg
      cucum

In [4]:
import os
from pathlib import Path

# Build class list from classname.txt (use train as reference)
with open("/content/kaggle_data/train/classname.txt", "r") as f:
    class_names = [line.strip() for line in f.readlines() if line.strip()]

class_map = {name: idx for idx, name in enumerate(class_names)}

print(f"Classes found: {class_names}")
print(f"Class map: {class_map}")

Classes found: ['aloevera', 'banana', 'bilimbi', 'cantaloupe', 'cassava', 'coconut', 'corn', 'cucumber', 'curcuma', 'eggplant', 'galangal', 'ginger', 'guava', 'kale', 'longbeans', 'mango', 'melon', 'orange', 'paddy', 'papaya', 'peperchili', 'pineapple', 'pomelo', 'shallot', 'soybeans', 'spinach', 'sweetpotatoes', 'tobacco', 'waterapple', 'watermelon']
Class map: {'aloevera': 0, 'banana': 1, 'bilimbi': 2, 'cantaloupe': 3, 'cassava': 4, 'coconut': 5, 'corn': 6, 'cucumber': 7, 'curcuma': 8, 'eggplant': 9, 'galangal': 10, 'ginger': 11, 'guava': 12, 'kale': 13, 'longbeans': 14, 'mango': 15, 'melon': 16, 'orange': 17, 'paddy': 18, 'papaya': 19, 'peperchili': 20, 'pineapple': 21, 'pomelo': 22, 'shallot': 23, 'soybeans': 24, 'spinach': 25, 'sweetpotatoes': 26, 'tobacco': 27, 'waterapple': 28, 'watermelon': 29}


In [5]:
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png"}

def generate_labels_for_split(split):
    split_dir = Path(split)
    label_count = 0
    missing_classes = []

    # Walk through each class subfolder
    for class_folder in split_dir.iterdir():
        if not class_folder.is_dir():
            continue

        class_name = class_folder.name  # e.g. "Mango"

        # Match case-insensitively against classname.txt
        matched_class = None
        for name in class_names:
            if name.lower() == class_name.lower():
                matched_class = name
                break

        if matched_class is None:
            missing_classes.append(class_name)
            print(f"  WARNING: '{class_name}' not found in classname.txt — skipping")
            continue

        class_id = class_map[matched_class]

        # Generate a .txt label for each image
        for img_path in class_folder.iterdir():
            if img_path.suffix.lower() not in IMG_EXTENSIONS:
                continue

            label_path = img_path.with_suffix(".txt")
            with open(label_path, "w") as f:
                f.write(f"{class_id} 0.5 0.5 1.0 1.0\n")

            label_count += 1

    print(f"[{split}] Labels generated: {label_count}")
    if missing_classes:
        print(f"[{split}] Skipped folders (not in classname.txt): {missing_classes}")

# Run for all 3 splits
for split in ["train", "val", "test"]:
    split = "/content/kaggle_data/"+split
    print(f"\nProcessing {split}...")
    generate_labels_for_split(split)



Processing /content/kaggle_data/train...
[/content/kaggle_data/train] Labels generated: 21000

Processing /content/kaggle_data/val...
[/content/kaggle_data/val] Labels generated: 3000

Processing /content/kaggle_data/test...
[/content/kaggle_data/test] Labels generated: 6000


In [6]:
# Spot check — print a few labels to confirm
for split in ["train", "val", "test"]:
    split = "/content/kaggle_data/"+split
    split_dir = Path(split)
    txt_files = list(split_dir.rglob("*.txt"))
    # Exclude classname.txt from count
    label_files = [f for f in txt_files if f.name != "classname.txt"]
    print(f"\n[{split}] Total label files: {len(label_files)}")

    # Print first 2 as sample
    for lf in label_files[:2]:
        print(f"  {lf} →", lf.read_text().strip())


[/content/kaggle_data/train] Total label files: 21000
  /content/kaggle_data/train/peperchili/peperchili207.txt → 20 0.5 0.5 1.0 1.0
  /content/kaggle_data/train/peperchili/peperchili316.txt → 20 0.5 0.5 1.0 1.0

[/content/kaggle_data/val] Total label files: 3000
  /content/kaggle_data/val/peperchili/peperchili735.txt → 20 0.5 0.5 1.0 1.0
  /content/kaggle_data/val/peperchili/peperchili789.txt → 20 0.5 0.5 1.0 1.0

[/content/kaggle_data/test] Total label files: 6000
  /content/kaggle_data/test/peperchili/peperchili813.txt → 20 0.5 0.5 1.0 1.0
  /content/kaggle_data/test/peperchili/peperchili960.txt → 20 0.5 0.5 1.0 1.0


In [7]:
import yaml

data_yaml = {
    "path": str("/content/kaggle_data"),   # root directory
    "train": "train",
    "val":   "val",
    "test":  "test",
    "nc":    len(class_names),
    "names": class_names
}

with open("data.yaml", "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("data.yaml created!")
print(data_yaml)

data.yaml created!
{'path': '/content/kaggle_data', 'train': 'train', 'val': 'val', 'test': 'test', 'nc': 30, 'names': ['aloevera', 'banana', 'bilimbi', 'cantaloupe', 'cassava', 'coconut', 'corn', 'cucumber', 'curcuma', 'eggplant', 'galangal', 'ginger', 'guava', 'kale', 'longbeans', 'mango', 'melon', 'orange', 'paddy', 'papaya', 'peperchili', 'pineapple', 'pomelo', 'shallot', 'soybeans', 'spinach', 'sweetpotatoes', 'tobacco', 'waterapple', 'watermelon']}


In [8]:
!ls -lrtha

total 28K
drwxr-xr-x 4 root root 4.0K Jun  4 13:39 .config
drwxr-xr-x 1 root root 4.0K Jun  4 13:39 sample_data
drwxr-xr-x 1 root root 4.0K Jun 12 07:57 ..
-rw-r--r-- 1 root root   68 Jun 12 08:15 kaggle.json
drwxr-xr-x 5 root root 4.0K Jun 12 08:15 kaggle_data
drwxr-xr-x 1 root root 4.0K Jun 12 08:15 .
-rw-r--r-- 1 root root  382 Jun 12 08:15 data.yaml


In [9]:
!cat data.yaml

names:
- aloevera
- banana
- bilimbi
- cantaloupe
- cassava
- coconut
- corn
- cucumber
- curcuma
- eggplant
- galangal
- ginger
- guava
- kale
- longbeans
- mango
- melon
- orange
- paddy
- papaya
- peperchili
- pineapple
- pomelo
- shallot
- soybeans
- spinach
- sweetpotatoes
- tobacco
- waterapple
- watermelon
nc: 30
path: /content/kaggle_data
test: test
train: train
val: val


In [10]:
!ls -lrtha /content/kaggle_data/train/mango/mango0.*

-rw-r--r-- 1 root root 53K Jun 12 08:15 /content/kaggle_data/train/mango/mango0.jpg
-rw-r--r-- 1 root root  19 Jun 12 08:15 /content/kaggle_data/train/mango/mango0.txt


In [11]:
!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")

model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="agri_detector",
    patience=10,
    augment=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False,